In [13]:
# ----------------------------------------
# 📊 Simple Bayesian Linear Regression for Media Mix Modeling (No Adstock)
# ----------------------------------------

import pymc as pm
import arviz as az
import pandas as pd
import numpy as np

import arviz_plots as azp

# ----------------------------------------
# Step 1: Simulate Media Spend and Sales Data
# ----------------------------------------

# Data in lakhs (TV & digital spend, and total sales)
data = {
    "sales":   [100, 120, 130, 125, 150, 160],  # observed weekly sales (lakhs)
    "tv":      [10, 15, 20, 15, 25, 30],        # TV media spend (lakhs)
    "digital": [5, 7, 10, 8, 12, 14]            # Digital media spend (lakhs)
}

df = pd.DataFrame(data)  # Create DataFrame for modeling

In [2]:
# ----------------------------------------
# Step 2: Define PyMC Bayesian Linear Model
# ----------------------------------------

with pm.Model() as model:

    # ---- Priors ----
    # These reflect our beliefs *before* seeing the data
    alpha = pm.Normal("intercept", mu=0, sigma=10)             # Intercept (baseline sales)
    beta_tv = pm.Normal("beta_tv", mu=0.2, sigma=0.1)           # TV channel effectiveness
    beta_digital = pm.Normal("beta_digital", mu=0.3, sigma=0.1) # Digital channel effectiveness
    sigma = pm.HalfNormal("sigma", sigma=10)                   # Standard deviation (uncertainty)

    # ---- Linear Model ----
    # Sales are modeled as a linear combination of TV and digital spend
    mu = alpha + beta_tv * df["tv"] + beta_digital * df["digital"]

    # ---- Likelihood ----
    # Observed sales are normally distributed around expected sales (mu)
    y_obs = pm.Normal("sales", mu=mu, sigma=sigma, observed=df["sales"])


In [19]:
import pymc as pm

# ----------------------------------------
# Step 3: Posterior Inference using MCMC
# ----------------------------------------

with model:
    trace = pm.sample(
        draws=200,
        tune=100,
        target_accept=0.9,
        random_seed=42
    )

# ----------------------------------------
# Step 4: Visualize Posterior Distributions
# ----------------------------------------
import arviz as az

az.plot_trace(
    trace,
    var_names=["beta_tv", "beta_digital"]
)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta_tv, beta_digital, sigma]


d:\Projects\MMM_projects\mmm_env\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 100 tune and 200 draw iterations (400 + 800 draws total) took 7 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


AttributeError: 'dict' object has no attribute '_repr_html_'